# IMT 2200 - Introducción a Ciencia de Datos  
**Pontificia Universidad Católica de Chile**  
**Instituto de Ingeniería Matemática y Computacional**  

# <h1><center>Interrogación 2 – Versión basada en PIB & OMS</center></h1>


## Instrucciones

* Esta parte de la Interrogación debe ser desarrollada completamente en lenguaje de programación **Python** en este Notebook de Jupyter.
* El desarrollo del Notebook debe ser claro y ordenado, incluyendo comentarios que permitan seguir fácilmente el código y los pasos implementados.
* Puedes usar las librerías **numpy**, **pandas**, **matplotlib** y **seaborn**, además de otras vistas en clases.
* Puedes agregar celdas adicionales donde lo estimes conveniente.
* Se evaluará tanto el **código** como las **respuestas e interpretaciones escritas**.
* **No se evaluarán respuestas sin código asociado.**

---

## Contexto de datos

En esta versión de la prueba analizaremos la relación entre el desarrollo económico y la salud de la población a nivel de país, usando dos bases de datos reales:

1. `pib2020-2025.csv`  
   - Contiene el Producto Interno Bruto (PIB) anual para distintos países entre los años 2020 y 2025.  
   - Columnas esperadas (entre otras):  
     * `Country`: nombre del país.  
     * `2020`, `2021`, `2022`, `2023`, `2024`, `2025`: valores de PIB por año.

2. `WHO_life_expectancy.csv`  
   - Contiene datos de la Organización Mundial de la Salud sobre esperanza de vida al nacer.  
   - Columnas típicas (pueden variar según versión, ajústelas si es necesario):  
     * `Location`: país.  
     * `Period`: año.  
     * `Dim1`: categoría (por ejemplo, sexo).  
     * `Value`: esperanza de vida al nacer, usualmente como texto con valor e intervalo.

En todas las preguntas, si los nombres de columnas difieren levemente de esta descripción, **adáptelos según el archivo real**.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# %matplotlib inline  # Descomentar si es necesario en tu entorno


---
## 1. Limpieza y transformación de las bases de datos  (25 pts)

En esta sección prepararemos ambas bases para poder analizarlas y combinarlas de forma consistente.


### (1.1) Carga inicial de datos  (*5 pts*)

Cargue los datos de los archivos `pib2020-2025.csv` y `WHO_life_expectancy.csv` en dos DataFrames llamados `pibData` y `lifeData`, respectivamente.

* Muestre las primeras filas de cada DataFrame.  
* Describa brevemente (2–3 líneas) su estructura: número de filas, columnas y tipos de datos más relevantes.


In [ ]:
# (1.1) Carga de datos y exploración inicial
pibData = pd.read_csv('pib2020-2025.csv')
lifeData = pd.read_csv('WHO_life_expectancy.csv')

pibData.head(), lifeData.head()

> Comentario: describa aquí la estructura general de ambos DataFrames.


### (1.2) Transformación de la base de PIB a formato largo  (*5 pts*)

A partir de `pibData`, genere un nuevo DataFrame llamado `pibLong` en formato “largo” con las siguientes columnas:

* `Country`: nombre del país.  
* `Year`: año (como entero).  
* `GDP`: valor del PIB para ese país y año.

Considere solo los años **2020 a 2025** (o el rango que efectivamente exista en el archivo).
Elimine las filas donde el valor de `GDP` sea nulo.


In [ ]:
# (1.2) Transformación wide → long para PIB
# Ajuste la lista de columnas de años según corresponda
year_cols = [c for c in pibData.columns if c.isdigit()]

pibLong = (
    pibData
    .melt(id_vars='Country', value_vars=year_cols,
          var_name='Year', value_name='GDP')
)

# Conversión de Year a entero y eliminación de nulos
pibLong['Year'] = pibLong['Year'].astype(int)
pibLong = pibLong.dropna(subset=['GDP'])

pibLong.head()

### (1.3) Limpieza de la base de esperanza de vida  (*7 pts*)

Prepare un DataFrame llamado `lifeClean` con las siguientes características:

* Considere solo las filas donde `Dim1` sea **"Both sexes"** (o equivalente en el archivo).  
* Considere únicamente el indicador de **esperanza de vida al nacer (años)** (ajuste según la columna que identifique el indicador si es necesario).  
* Renombre columnas para trabajar de forma homogénea:
  * `Location` → `Country`  
  * `Period` → `Year`
* La columna `Value` suele tener un texto del tipo `"72.5 [71.3-73.6]"`.  
  Extraiga el número inicial (antes del espacio) y conviértalo a un valor numérico (float) en una nueva columna `LifeExpectancy`.

Elimine las filas donde `LifeExpectancy` no se pueda interpretar numéricamente o sea nula.


In [ ]:
# (1.3) Limpieza y transformación de la base de esperanza de vida

lifeClean = lifeData.copy()

# Filtrado de 'Both sexes' si la columna Dim1 existe
if 'Dim1' in lifeClean.columns:
    lifeClean = lifeClean[lifeClean['Dim1'] == 'Both sexes']

# Renombrar columnas principales si existen
rename_map = {}
if 'Location' in lifeClean.columns:
    rename_map['Location'] = 'Country'
if 'Period' in lifeClean.columns:
    rename_map['Period'] = 'Year'
lifeClean = lifeClean.rename(columns=rename_map)

# Extraer valor numérico de la columna 'Value'
def extract_value(x):
    if isinstance(x, str):
        try:
            return float(x.split(' ')[0])
        except:
            return np.nan
    return np.nan

lifeClean['LifeExpectancy'] = lifeClean['Value'].apply(extract_value)
lifeClean = lifeClean.dropna(subset=['LifeExpectancy'])

# Asegurarse de que Year sea entero si existe
if 'Year' in lifeClean.columns:
    lifeClean['Year'] = lifeClean['Year'].astype(int)

lifeClean.head()

### (1.4) Selección de países en común  (*4 pts*)

Para evitar problemas de nombres distintos entre bases de datos, trabajaremos solo con los países que aparecen **en ambas**.

1. Obtenga el conjunto de países de `pibLong` y de `lifeClean`.  
2. Calcule la intersección de ambos conjuntos.  
3. Genere dos DataFrames filtrados:
   * `pibCommon`: subconjunto de `pibLong` con solo los países en común.  
   * `lifeCommon`: subconjunto de `lifeClean` con solo los países en común.


In [ ]:
# (1.4) Filtrado de países en común entre ambas bases

pib_countries = set(pibLong['Country'].unique())
life_countries = set(lifeClean['Country'].unique())

common_countries = pib_countries.intersection(life_countries)
len(common_countries), list(sorted(common_countries))[:10]  # muestra algunos

pibCommon = pibLong[pibLong['Country'].isin(common_countries)].copy()
lifeCommon = lifeClean[lifeClean['Country'].isin(common_countries)].copy()

pibCommon.head(), lifeCommon.head()

### (1.5) Base consolidada `socioData`  (*4 pts*)

Usando `pibCommon` y `lifeCommon`, construya un DataFrame consolidado llamado `socioData` que contenga, para cada combinación país–año:

* `Country`  
* `Year`  
* `GDP`  
* `LifeExpectancy`  

Para ello, realice un `merge` adecuado entre ambas bases sobre las columnas `Country` y `Year`.

Verifique que:

* No existan filas duplicadas por combinación `Country`–`Year`.  
* No existan valores nulos en `GDP` ni en `LifeExpectancy`.

En caso de encontrarlos, explique brevemente cómo los maneja (eliminar filas, imputar, etc.).


In [ ]:
# (1.5) Construcción de la base consolidada socioData

socioData = pd.merge(
    pibCommon,
    lifeCommon[['Country', 'Year', 'LifeExpectancy']],
    on=['Country', 'Year'],
    how='inner'
)

# Verificar duplicados
dup_mask = socioData.duplicated(subset=['Country', 'Year'], keep=False)
socioData[dup_mask].head()

# Eliminar filas con nulos en las variables clave
socioData = socioData.dropna(subset=['GDP', 'LifeExpectancy'])

socioData.head(), socioData.shape

> Comentario: describa si encontró duplicados o nulos y qué decisión tomó al respecto.


---
## 2. Análisis exploratorio de datos (EDA)  (25 pts)

En esta sección exploraremos la relación entre el PIB y la esperanza de vida usando la base consolidada `socioData`.


### (2.1) Año de referencia  (*5 pts*)

Para hacer comparaciones claras entre países, trabajaremos con un año fijo.

1. Verifique qué años están disponibles en la columna `Year` de `socioData`.  
2. Seleccione el año **2023** si está disponible; en caso contrario, seleccione el año más reciente disponible.  
3. Genere un DataFrame `socioRef` que contenga solo las observaciones de ese año seleccionado.

Muestre cuántos países hay en `socioRef` y las primeras filas.


In [ ]:
# (2.1) Selección de año de referencia

available_years = sorted(socioData['Year'].unique())
print('Años disponibles en socioData:', available_years)

ref_year = 2023 if 2023 in available_years else max(available_years)
print('Año de referencia seleccionado:', ref_year)

socioRef = socioData[socioData['Year'] == ref_year].copy()
socioRef.shape, socioRef.head()

### (2.2) Correlación y visualización básica  (*10 pts*)

Sobre el DataFrame `socioRef`:

1. Genere una **matriz de correlación** que incluya al menos las variables `GDP` y `LifeExpectancy`. Muéstrela como un **heatmap**.  
2. Construya un **diagrama de dispersión (scatterplot)** con:
   * eje X = `GDP`  
   * eje Y = `LifeExpectancy`  

   Si lo considera adecuado, puede usar escala logarítmica para el eje X.

Comente brevemente (3–4 líneas) qué tipo de relación parece observarse entre estas variables.


In [ ]:
# (2.2) Heatmap de correlación y scatterplot

corr = socioRef[['GDP', 'LifeExpectancy']].corr()

plt.figure()
sns.heatmap(corr, annot=True, cmap='viridis')
plt.title('Correlación entre PIB y esperanza de vida')
plt.show()

plt.figure()
sns.scatterplot(data=socioRef, x='GDP', y='LifeExpectancy')
plt.title(f'PIB vs Esperanza de vida ({ref_year})')
plt.xlabel('PIB')
plt.ylabel('Esperanza de vida')
plt.show()

> Comentario: describa brevemente la relación observada entre PIB y esperanza de vida para el año de referencia.


### (2.3) Comparación entre países extremos  (*10 pts*)

Usando `socioRef`:

1. Ordene los países de mayor a menor según `GDP`.  
2. Muestre una tabla con los **5 países con mayor PIB** y otra con los **5 países con menor PIB** (si hay suficientes países).  
3. Compare los niveles de `LifeExpectancy` entre estos grupos.

Escriba un comentario (3–5 líneas) indicando si los países con mayor PIB presentan sistemáticamente una mayor esperanza de vida que los de menor PIB.


In [ ]:
# (2.3) Comparación de países según PIB

socioRef_sorted = socioRef.sort_values('GDP', ascending=False)

top5 = socioRef_sorted.head(5)
bottom5 = socioRef_sorted.tail(5)

top5, bottom5

> Comentario: compare aquí la esperanza de vida entre los países con mayor y menor PIB.


---
## 3. Modelando la relación entre PIB y esperanza de vida  (30 pts)

En esta sección utilizaremos modelos de regresión para estudiar la relación entre el PIB de un país y su esperanza de vida, usando todos los años disponibles en `socioData`.


### (3.1) Preparación de los datos para modelar  (*8 pts*)

Trabajaremos con todas las filas de `socioData`.

1. Defina como variable dependiente `y` la columna `LifeExpectancy`.  
2. Como variable independiente principal, utilizaremos el PIB:
   * Cree una nueva columna `log_GDP = log(GDP)` para estabilizar la escala.  
3. Use como predictores al menos:
   * `log_GDP`  
   * `Year`  

4. Divida los datos en conjuntos de **entrenamiento** y **prueba**, usando una proporción cercana a 75%/25%.  
   Puede utilizar funciones de la librería `sklearn` si fue vista en clases, o construir el split manualmente.

Muestre el tamaño de cada conjunto.


In [ ]:
# (3.1) Preparación de datos para modelar

import math
from sklearn.model_selection import train_test_split

socioData = socioData.copy()
socioData['log_GDP'] = np.log(socioData['GDP'])

X = socioData[['log_GDP', 'Year']]
y = socioData['LifeExpectancy']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

X_train.shape, X_test.shape

### (3.2) Regresión lineal múltiple  (*10 pts*)

Entrene un modelo de **regresión lineal múltiple** usando los predictores `log_GDP` y `Year`.

1. Ajuste el modelo con el conjunto de entrenamiento.  
2. Calcule el **RMSE** (error cuadrático medio raíz) y el **R²** en los conjuntos de entrenamiento y prueba.  
3. Comente si el modelo captura razonablemente bien la relación entre PIB, año y esperanza de vida.

> Puede usar `LinearRegression` de `sklearn` u otra herramienta vista en el curso.


In [ ]:
# (3.2) Modelo de regresión lineal múltiple

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

linreg = LinearRegression()
linreg.fit(X_train, y_train)

y_train_pred = linreg.predict(X_train)
y_test_pred = linreg.predict(X_test)

rmse_train = math.sqrt(mean_squared_error(y_train, y_train_pred))
rmse_test = math.sqrt(mean_squared_error(y_test, y_test_pred))
r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

rmse_train, rmse_test, r2_train, r2_test

> Comentario: evalúe brevemente la capacidad predictiva del modelo lineal múltiple.


### (3.3) Extensión del modelo y discusión  (*12 pts*)

Considere ahora extender el modelo incorporando alguna transformación adicional o interacción, por ejemplo:

* Un término cuadrático sobre `log_GDP` (`(log_GDP)^2`).  
* Una interacción entre `log_GDP` y `Year`.  

1. Construya un nuevo set de predictores agregando al menos una de estas extensiones.  
2. Entrene un nuevo modelo de regresión lineal con estos predictores extendidos.  
3. Calcule nuevamente RMSE y R² en entrenamiento y prueba.  
4. Compare los resultados del modelo extendido con el modelo de la parte (3.2).

Responda en 4–5 líneas:

* ¿Mejora la calidad del ajuste al agregar estos términos?  
* ¿Observa riesgo de sobreajuste?  
* ¿Qué modelo preferiría usar para interpretar la relación entre PIB y esperanza de vida?


In [ ]:
# (3.3) Modelo extendido con término cuadrático e interacción

X_ext = X.copy()
X_ext['log_GDP_sq'] = X_ext['log_GDP']**2
X_ext['log_GDP_Year'] = X_ext['log_GDP'] * X_ext['Year']

X_train_ext, X_test_ext, y_train_ext, y_test_ext = train_test_split(
    X_ext, y, test_size=0.25, random_state=42
)

linreg_ext = LinearRegression()
linreg_ext.fit(X_train_ext, y_train_ext)

y_train_pred_ext = linreg_ext.predict(X_train_ext)
y_test_pred_ext = linreg_ext.predict(X_test_ext)

rmse_train_ext = math.sqrt(mean_squared_error(y_train_ext, y_train_pred_ext))
rmse_test_ext = math.sqrt(mean_squared_error(y_test_ext, y_test_pred_ext))
r2_train_ext = r2_score(y_train_ext, y_train_pred_ext)
r2_test_ext = r2_score(y_test_ext, y_test_pred_ext)

rmse_train_ext, rmse_test_ext, r2_train_ext, r2_test_ext

> Comentario: compare aquí el modelo extendido con el modelo base y discuta cuál usaría y por qué.


---

## Fin de la Interrogación 2 – Versión PIB & OMS

Guarde este Notebook y entréguelo según las instrucciones del curso.
